# VoxCPM2 server on Kaggle (free, ~30 GPU hours per week)

1. In the right-hand panel → **Session options**: set **Accelerator** to **GPU T4 x2** and turn **Internet** on.
   (Kaggle asks you to verify your phone number once before GPU and Internet can be used.)
2. **Run All**. The last cell prints a `https://….trycloudflare.com` URL — paste it into the app's **Settings → Server URL**.
3. Keep this tab open while you use the app. When you're done, click **Stop session** (power icon) so your weekly GPU hours aren't used while idle.

In [ ]:
!nvidia-smi -L
!pip -q install voxcpm fastapi "uvicorn[standard]" python-multipart soundfile librosa
!wget -q -O server.py https://raw.githubusercontent.com/yassineallach/admin/claude/android-voxcpm-voice-app-8pkl1u/voxcpm-app/server/server.py
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
import os
# Optional: protect the server. Put the same key in the app's Settings.
os.environ['VOXCPM_API_KEY'] = ''

import subprocess, time, urllib.request, urllib.error
server = subprocess.Popen(['python', 'server.py', '--port', '8000'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(600):
    if server.poll() is not None:
        break  # crashed, the log below says why
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2); break
    except urllib.error.HTTPError:
        break  # answering (401 just means an API key is set)
    except Exception:
        time.sleep(2)
print(open('server.log').read()[-2000:])

In [ ]:
import re, subprocess, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.DEVNULL, stderr=open('tunnel.log', 'w'))
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        print('Server URL for the app:', m.group(0)); break
    time.sleep(1)
else:
    print(open('tunnel.log').read())